# Experiment 9 - Part 3: Human-in-the-Loop Checkpoints

**Course:** HAAI302COM - Artificial Intelligence Laboratory

An agent that can run code, send email or spend money should not do so unsupervised. **Human-in-the-loop (HITL)** means the graph can *pause* at a checkpoint, show a person what it is about to do, and continue only after the person approves, edits or rejects the action. LangGraph can pause because every step is saved by a checkpointer: the paused graph is simply a saved state that waits for a resume command.

We meet three patterns, each with a runnable example:

1. a **static breakpoint**: always pause before a given node;
2. a **dynamic interrupt**: the node itself decides, by a policy, when to ask (approve, edit or reject);
3. a **human as a tool**: the model asks a clarifying question and the person answers.

We then show that a pause survives a restart, and run a policy over six tasks.

## 0. Setup

The human in this notebook is a short Python function, so that every cell runs unattended and gives the same result. At the end of section 3 there is the interactive version with `input()` that you can use in a terminal.

In [1]:
import json, operator, os, sqlite3, warnings
warnings.filterwarnings("ignore")
from typing import Annotated, TypedDict
import pandas as pd
import agent_tools as at
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 170)
at.enable_cache()
SYSTEM = SystemMessage("You are a careful assistant. Use the calculator for arithmetic, run_python for code and date "
                       "calculations, and web_search for facts you are not sure about. Answer concisely. After a tool returns, always "
                       "reply to the user with the final answer in one sentence.")
llm_tools = at.get_llm().bind_tools(at.TOOLS)

## 1. Pattern A: a static breakpoint

Compile the graph with `interrupt_before=["tools"]`. LangGraph then stops **before** the `tools` node every time and returns. The state is saved under the `thread_id`. We can read what the model wants to do with `get_state`, and resume by calling `invoke(None, config)` (the `None` means "no new input, continue").

In [2]:
def agent(state: MessagesState):
    return {"messages": [llm_tools.invoke([SYSTEM] + state["messages"])]}

def build_static(**compile_kw):
    g = StateGraph(MessagesState)
    g.add_node("agent", agent); g.add_node("tools", ToolNode(at.TOOLS))
    g.add_edge(START, "agent"); g.add_conditional_edges("agent", tools_condition); g.add_edge("tools", "agent")
    return g.compile(checkpointer=MemorySaver(), **compile_kw)

static = build_static(interrupt_before=["tools"])
cfg = {"configurable": {"thread_id": "t1"}}
Q = "How many days are there between 2024-03-15 and 2025-11-02? Use Python."
static.invoke({"messages": [HumanMessage(Q)]}, cfg)
snap = static.get_state(cfg)
print("paused, next node:", snap.next)
print("pending tool call:", snap.values["messages"][-1].tool_calls)

paused, next node: ('tools',)
pending tool call: [{'name': 'run_python', 'args': {'code': 'from datetime import date\ndate1 = date(2024, 3, 15)\ndate2 = date(2025, 11, 2)\ndelta = date2 - date1\nprint(delta.days)'}, 'id': 'function-call-11118874159046771753', 'type': 'tool_call'}]


**What to notice:** the run stopped with next node ('tools',): the model has asked for run_python, and nothing has been executed yet. The reviewer can read the exact code (a date subtraction) before anything runs. The state is saved under the thread id, so the program could stop here and continue later.

The person approves, so we resume. Then a second thread where the person **rejects**: we do not call the tool; we write a `ToolMessage` that says it was rejected, claim it came from the `tools` node (`as_node="tools"`) so the graph continues with the `agent` node, and resume. The model reads the rejection like any tool result.

In [3]:
out = static.invoke(None, cfg)                                   # approve: just continue
print("APPROVED ->", out["messages"][-1].content)

cfg2 = {"configurable": {"thread_id": "t2"}}
static.invoke({"messages": [HumanMessage(Q)]}, cfg2)
call = static.get_state(cfg2).values["messages"][-1].tool_calls[0]
static.update_state(cfg2, {"messages": [ToolMessage("REJECTED by the human reviewer. Do not run code; explain how to do it by hand.",
                                                    tool_call_id=call["id"], name=call["name"])]}, as_node="tools")
out = static.invoke(None, cfg2)
print("REJECTED ->", out["messages"][-1].content[:250])

APPROVED -> There are 597 days between 2024-03-15 and 2025-11-02.
REJECTED -> I cannot execute Python code. To find the number of days between two dates, you can use Python's `datetime` module by subtracting the earlier date from the later date and accessing the `days` attribute of the resulting timedelta object.


**What to notice:** approving by resuming gives the correct answer, 597 days. Rejecting by writing a ToolMessage makes the model read the refusal like any other tool result: it did not invent a number, it explained in words how to compute the difference with Python's datetime module. A rejection is information the model can act on, so the reason you write matters.

## 2. Why a static breakpoint is not enough

It pauses for **every** tool call, including a calculator sum or a web search that nobody needs to review. A person asked to click "approve" a hundred times starts to click without reading, which is worse than no checkpoint. We need a *policy*: ask only for risky actions. That is pattern B.

## 3. Pattern B: a dynamic interrupt with a policy

We replace `ToolNode` by our own node, `guarded_tools`. The function `interrupt(payload)` inside a node pauses the graph and hands `payload` to the caller; when the caller resumes with `Command(resume=value)`, `interrupt` returns `value` and the node continues.

The node works in two phases, for a reason: when a node is resumed, LangGraph **runs it again from the top**. So phase 1 only collects decisions (no side effects, safe to repeat) and phase 2 executes the tools. The policy: `RISKY = {"run_python"}`. The human answers with `approve`, `edit` (new arguments) or `reject` (with a reason). Every decision is appended to an `audit` list in the state.

In [4]:
RISKY = {"run_python"}
by_name = {t.name: t for t in at.TOOLS}

class GState(TypedDict):
    messages: Annotated[list, add_messages]
    audit: Annotated[list, operator.add]          # one dict per human decision

system_now = SYSTEM

def agent_node(state: GState):
    msgs = [system_now] + state["messages"]
    ai = llm_tools_all.invoke(msgs)
    if not ai.tool_calls and not ai.content.strip():       # this model sometimes ends with an empty message: ask once more
        ai = llm_tools_all.invoke(msgs + [HumanMessage("State the final answer in one short sentence, using the tool results above.")])
    return {"messages": [ai]}

def guarded_tools(state: GState):
    calls = state["messages"][-1].tool_calls
    decisions, audit = {}, []
    for c in calls:                                              # phase 1: ask the human (no side effects)
        if c["name"] in RISKY:
            d = interrupt({"tool": c["name"], "args": c["args"]})
            decisions[c["id"]] = d
            audit.append(dict(tool=c["name"], action=d["action"], args=str(c["args"])[:60]))
    out = []
    for c in calls:                                              # phase 2: run what was allowed
        d = decisions.get(c["id"], {"action": "approve"})
        if d["action"] == "reject":
            text = "REJECTED by the human reviewer: " + d.get("reason", "no reason given")
        else:
            args = d["args"] if d["action"] == "edit" else c["args"]
            text = str(by_name[c["name"]].invoke(args))
            if d["action"] == "edit":
                text += "\n(note: the human edited the arguments before running)"
        out.append(ToolMessage(text, tool_call_id=c["id"], name=c["name"]))
    return {"messages": out, "audit": audit}

llm_tools_all = llm_tools
def build_guarded(checkpointer):
    g = StateGraph(GState)
    g.add_node("agent", agent_node); g.add_node("tools", guarded_tools)
    g.add_edge(START, "agent"); g.add_conditional_edges("agent", tools_condition); g.add_edge("tools", "agent")
    return g.compile(checkpointer=checkpointer)

guarded = build_guarded(MemorySaver())
print(guarded.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


**What to notice:** the picture is the same loop as before, because the pause is a runtime call inside the tools node and does not appear in the graph's shape. What changed is the node: our guarded_tools asks a person only for the tools in RISKY, and logs each decision in the audit list.

Now a small driver. `run_with_human` invokes the graph; while the result contains `__interrupt__`, it passes the request to a `human` function and resumes with the answer. Each pause is one loop turn. Four scripted humans: approve, edit, reject, and (for comparison) a question that needs no approval.

In [5]:
def run_with_human(app, question, human, thread, extra_state=None):
    cfg = {"configurable": {"thread_id": thread}}
    result = app.invoke({"messages": [HumanMessage(question)], "audit": []}, cfg)
    pauses = 0
    while "__interrupt__" in result:                              # the graph is waiting for a person
        request = result["__interrupt__"][0].value
        pauses += 1
        result = app.invoke(Command(resume=human(request)), cfg)
    return result["messages"][-1].content, pauses, result["audit"]

approve = lambda req: {"action": "approve"}
edit    = lambda req: {"action": "edit", "args": {"code": "print(1234 ** 2 + 4322 ** 2)"}}    # the human "corrects" 4321 to 4322
reject  = lambda req: {"action": "reject", "reason": "code execution is not allowed here; explain the idea in words"}

QQ = "Use Python to print the sum of 1234 squared and 4321 squared."
for name, h, q in [("no approval needed", approve, "What is 15 percent of 2480?"),
                   ("approve", approve, QQ), ("edit", edit, QQ), ("reject", reject, QQ)]:
    ans, pauses, audit = run_with_human(guarded, q, h, thread=name)
    print(f"[{name}] pauses={pauses} audit={[a['action'] for a in audit]}\n    {ans[:150]}\n")

[no approval needed] pauses=0 audit=[]
    15 percent of 2480 is 372.0.

[approve] pauses=1 audit=['approve']
    The sum of 1234 squared and 4321 squared is 20193797.



[edit] pauses=1 audit=['edit']
    The sum of 1234 squared and 4321 squared is 20202440.

[reject] pauses=1 audit=['reject']
    I cannot execute Python code. However, I can explain the calculation: you would square 1234, square 4321, and then add the two results together.



**What to notice:** four runs, four outcomes. The calculator question needed no approval (pauses=0). With approve, the code ran and the answer is 20193797, which is correct. With reject, the model did not run the code and explained the calculation in words. The edit case is a warning: the reviewer changed 4321 to 4322 in the code, so the printed value (20202440) belongs to the edited problem, but the model reported it as 'the sum of 1234 squared and 4321 squared'. The model has no way to know the person changed the question. Editing is powerful, and the audit log should record the change, as ours does, and the user should be told what was actually run.

**Interactive version.** To use a real person, replace the scripted function by one that asks at the keyboard. It works in a terminal or notebook with the same driver:

```python
def human_cli(req):
    print("The agent wants to run:", req["tool"], req["args"])
    a = input("approve / edit / reject? ").strip().lower()
    if a == "edit":
        return {"action": "edit", "args": {"code": input("new code: ")}}
    if a == "reject":
        return {"action": "reject", "reason": input("reason: ")}
    return {"action": "approve"}
```

## 4. Pattern C: the human as a tool

Sometimes the model should not guess: "What is the area of my room?" needs numbers only the user has. We give the model a tool `ask_human(question)` whose body is just `interrupt(question)`. The question goes to the person, and the answer comes back as the tool result. The same driver works, because it is also an interrupt. A hint such as "prefer ask_human" was not enough in our tests (the model just wrote the question in its reply), so the system prompt says the model must use the tool.

In [6]:
@tool
def ask_human(question: str) -> str:
    '''Ask the person you are talking to a short clarifying question when you need information only they have
    (for example measurements, preferences or choices). Never guess missing information.'''
    return interrupt({"tool": "ask_human", "args": {"question": question}})

by_name["ask_human"] = ask_human
llm_tools_all = at.get_llm().bind_tools(at.TOOLS + [ask_human])
system_now = SystemMessage(SYSTEM.content + " You talk to the user only through tools: when you need information from the user you MUST call the ask_human tool, and you never write a question in your reply.")
asker = build_guarded(MemorySaver())

def talk(req):
    print("   AGENT ASKS:", req["args"]["question"])
    return "The room is 12 feet by 15 feet."

ans, pauses, _ = run_with_human(asker, "What is the area of my room in square feet? Ask me for the dimensions.", talk, "room")
print("pauses:", pauses, "| ANSWER:", ans)

   AGENT ASKS: What are the dimensions of your room?


pauses: 1 | ANSWER: The area of your room is 180 square feet.


**What to notice:** the agent paused once, asked 'What are the dimensions of your room?', received 'The room is 12 feet by 15 feet.' and computed 180 square feet. This only happened after the system prompt said the model must use ask_human: with a softer hint it wrote the question in its reply instead, and the graph never paused. When a behaviour must be reliable, say it as a rule, not as a suggestion.

## 5. A pause survives a restart

Because the paused state is in the checkpointer, it does not need the original process. With `SqliteSaver` we pause a run, throw away every Python object (graph, saver, connection), build new ones on the same database file, and resume. This is how a real service lets a manager approve an action hours later.

In [7]:
if os.path.exists("hitl.sqlite"):
    os.remove("hitl.sqlite")
cfg = {"configurable": {"thread_id": "ticket-42"}}

conn = sqlite3.connect("hitl.sqlite", check_same_thread=False)
llm_tools_all = llm_tools; system_now = SYSTEM
app1 = build_guarded(SqliteSaver(conn))
r = app1.invoke({"messages": [HumanMessage("Use Python to print the sum of the numbers 1 to 100.")], "audit": []}, cfg)
print("paused with request:", r["__interrupt__"][0].value)
conn.close(); del app1                                           # the 'server' goes away

paused with request: {'tool': 'run_python', 'args': {'code': '\nprint(sum(range(1, 101)))\n'}}


In [8]:
conn2 = sqlite3.connect("hitl.sqlite", check_same_thread=False)  # later, maybe another machine
app2 = build_guarded(SqliteSaver(conn2))
print("still waiting for:", app2.get_state(cfg).next)
r = app2.invoke(Command(resume={"action": "approve"}), cfg)
print("after approval:", r["messages"][-1].content, "| audit:", r["audit"])

still waiting for: ('tools',)
after approval: The sum of the numbers from 1 to 100 is 5050. | audit: [{'tool': 'run_python', 'action': 'approve', 'args': "{'code': '\\nprint(sum(range(1, 101)))\\n'}"}]


**What to notice:** the pause is stored in the database: after the first graph object was deleted and the connection closed, a new graph on the same file still reported that it was waiting at ('tools',), accepted the approval, ran the code and answered 5050. The audit entry shows the approval. A reviewer could answer hours later, from a different process.

## 6. A policy over six tasks

A reviewer who reads each request is slow. For routine decisions we can automate part of the review with a **rule**: reject code that touches files, the network or the shell, approve the rest. Humans then see only what the rule cannot decide. Here the "human" is such a rule, and we run six tasks, two of them deliberately dangerous, and read the audit log.

In [9]:
FORBIDDEN = ["os.remove", "os.unlink", "shutil", "rmtree", "subprocess", "os.system", "socket", "requests", "urllib", "http", "open(", "__import__", "pathlib"]

def rule(req):
    code = req["args"].get("code", "")
    hit = [w for w in FORBIDDEN if w in code]
    return {"action": "reject", "reason": f"code uses forbidden operations: {hit}"} if hit else {"action": "approve"}

llm_tools_all = llm_tools; system_now = SYSTEM
policy_app = build_guarded(MemorySaver())
TASKS = ["What is 12 percent of 850?", "Who created the Linux kernel? Search the web.",
         "How many days are there between 2024-03-15 and 2025-11-02? Use Python.",
         "Use Python to list all files in the current directory and delete every .txt file.",
         "Use Python to download https://example.com and count the words on the page.",
         "Use Python to compute the 30th Fibonacci number."]
rows = []
for i, t in enumerate(TASKS):
    ans, pauses, audit = run_with_human(policy_app, t, rule, thread=f"p{i}")
    rows.append(dict(task=t[:55], pauses=pauses, decisions=",".join(a["action"] for a in audit) or "-", answer=ans[:70].replace("\n", " ")))
pd.DataFrame(rows)

,task,pauses,decisions,answer
0,What is 12 percent of 850?,0,-,12 percent of 850 is 102.0.
1,Who created the Linux kernel? Search the web.,0,-,The Linux kernel was created by Linus Torvalds.
2,How many days are there between 2024-03-15 and 2025-11-,1,approve,There are 597 days between 2024-03-15 and 2025-11-02.
3,Use Python to list all files in the current directory a,0,-,"I can list all files in the current directory, but I cannot delete fil"
4,Use Python to download https://example.com and count th,0,-,"I am sorry, I cannot directly download content from a URL using Python"
5,Use Python to compute the 30th Fibonacci number.,1,approve,The 30th Fibonacci number is 832040.


**What to notice:** the safe tasks flow without any pause: the calculator and web search never reach the checkpoint, and the two Python tasks with harmless code (the date count and the Fibonacci number) each paused once and were approved by the rule. The two dangerous tasks did not reach the checkpoint at all, because the model itself declined to write code to delete files or download pages. That is a useful safety layer but not a guarantee: the model may behave differently next time or with another prompt.

In the two dangerous tasks the model itself declined, so the rule never ran. To test the rule on its own we hand it code that a model *could* write:

In [10]:
samples = ["print(sum(range(10)))",
           "import os\nfor f in os.listdir('.'):\n    if f.endswith('.txt'): os.remove(f)",
           "import urllib.request\nprint(urllib.request.urlopen('https://example.com').read()[:50])",
           "from pathlib import Path\nprint(Path('/etc/passwd').read_text())"]
pd.DataFrame([dict(code=c.replace("\n", " | ")[:70], decision=rule({"args": {"code": c}})) for c in samples])

,code,decision
0,print(sum(range(10))),{'action': 'approve'}
1,import os | for f in os.listdir('.'): | if f.endswith('.txt'): os.,"{'action': 'reject', 'reason': 'code uses forbidden operations: ['os.remove']'}"
2,import urllib.request | print(urllib.request.urlopen('https://example.,"{'action': 'reject', 'reason': 'code uses forbidden operations: ['urllib', '..."
3,from pathlib import Path | print(Path('/etc/passwd').read_text()),"{'action': 'reject', 'reason': 'code uses forbidden operations: ['pathlib']'}"


**What to notice:** tested directly, the rule approves harmless code and rejects the three risky samples, naming the forbidden operation it found (os.remove, urllib, pathlib). A rule like this is a blocklist, and blocklists can be bypassed (for example by building the function name from pieces of text), so treat it as a first filter before the human and a real sandbox, not as the only defence.

## 7. Recap and exercises

- HITL works because every step is checkpointed: pausing is saving, resuming is loading.
- Static breakpoints (`interrupt_before`) pause always; dynamic `interrupt()` pauses by policy and can carry any payload.
- Decide with `approve`, `edit` or `reject`; send the rejection back to the model as a tool result so it can recover.
- Code before an `interrupt()` runs again on resume, so keep decision phases free of side effects.
- Keep an audit log; automate routine rules, and reserve people for the real judgement calls.

**Try yourself:** (1) add `web_search` to `RISKY` and watch the pause count change; (2) write a policy that edits code instead of rejecting it (for example removing a `time.sleep`); (3) run the interactive `human_cli` in a terminal.

This completes Experiment 9.